# Part 2 Notebook - EDA Business Story (Datathon 2026)

Goal: tell a complete business story with **hard numbers** and **actionable interpretations**.

This notebook exports:
- KPI tables (`CSV`)
- charts (`PNG`)
- executive summary (`Markdown` + `JSON`)
to `../eda_results/scoring/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from IPython.display import display, Markdown

DATA_DIR = Path('../dataset')
OUT_DIR = Path('../eda_results/scoring')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid')

sales = pd.read_csv(DATA_DIR / 'sales.csv', parse_dates=['Date']).sort_values('Date')
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_DIR / 'order_items.csv', low_memory=False)
products = pd.read_csv(DATA_DIR / 'products.csv')
customers = pd.read_csv(DATA_DIR / 'customers.csv', parse_dates=['signup_date'])
geography = pd.read_csv(DATA_DIR / 'geography.csv')
returns = pd.read_csv(DATA_DIR / 'returns.csv', parse_dates=['return_date'])
payments = pd.read_csv(DATA_DIR / 'payments.csv')
shipments = pd.read_csv(DATA_DIR / 'shipments.csv', parse_dates=['ship_date', 'delivery_date'])
web = pd.read_csv(DATA_DIR / 'web_traffic.csv', parse_dates=['date'])
promotions = pd.read_csv(DATA_DIR / 'promotions.csv', parse_dates=['start_date', 'end_date'])
reviews = pd.read_csv(DATA_DIR / 'reviews.csv', parse_dates=['review_date'])
inventory = pd.read_csv(DATA_DIR / 'inventory.csv', parse_dates=['snapshot_date'])

line = order_items.copy()
line['gross'] = line['quantity'] * line['unit_price']
line['net'] = line['gross'] - line['discount_amount']
line['has_promo'] = line['promo_id'].notna()
line = line.merge(products[['product_id', 'cogs', 'segment', 'category', 'size']], on='product_id', how='left')
line['cost_est'] = line['quantity'] * line['cogs']
line['margin_est'] = line['net'] - line['cost_est']
line['margin_est_pct'] = np.where(line['net'] > 0, line['margin_est'] / line['net'], np.nan)
print('Data prepared.')

## 1) Financial Trajectory: Revenue, Cost, Margin

In [ ]:
sales['year'] = sales['Date'].dt.year
sales['month'] = sales['Date'].dt.month
sales['dow_name'] = sales['Date'].dt.day_name()

total_revenue = float(sales['Revenue'].sum())
total_cogs = float(sales['COGS'].sum())
gross_margin_pct = float((total_revenue - total_cogs) / total_revenue * 100)
n_years = (sales['Date'].max() - sales['Date'].min()).days / 365.25
cagr = float((sales.iloc[-1]['Revenue'] / sales.iloc[0]['Revenue']) ** (1 / n_years) - 1)

yearly = sales.groupby('year').agg(revenue=('Revenue', 'sum'), cogs=('COGS', 'sum'))
yearly['margin_pct'] = (yearly['revenue'] - yearly['cogs']) / yearly['revenue'] * 100
yearly['yoy_pct'] = yearly['revenue'].pct_change() * 100
display(yearly.round(2))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
yearly['revenue'].plot(kind='bar', ax=axes[0], color='#1f77b4')
axes[0].set_title('Yearly Revenue')
axes[0].set_ylabel('Revenue')
yearly['margin_pct'].plot(kind='line', marker='o', ax=axes[1], color='#d62728')
axes[1].set_title('Yearly Estimated Gross Margin %')
axes[1].set_ylabel('Margin %')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_story_01_yearly_revenue_margin.png', dpi=150)
plt.show()

print(f'Total revenue: {total_revenue:,.0f}')
print(f'Total COGS: {total_cogs:,.0f}')
print(f'Gross margin: {gross_margin_pct:.2f}%')
print(f'Revenue CAGR (daily endpoints): {cagr * 100:.2f}%')

## 2) Seasonality: Month, Weekday, Holiday Windows

In [ ]:
month_avg = sales.groupby('month')['Revenue'].mean().sort_values(ascending=False)
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_avg = sales.groupby('dow_name')['Revenue'].mean().reindex(dow_order)

years = range(sales['year'].min(), sales['year'].max() + 1)
vn_holidays = holidays.VN(years=years)
sales['is_holiday'] = sales['Date'].dt.date.isin(vn_holidays)
sales['holiday_name'] = sales['Date'].dt.date.map(vn_holidays)
holiday_mean = float(sales.loc[sales['is_holiday'], 'Revenue'].mean())
non_holiday_mean = float(sales.loc[~sales['is_holiday'], 'Revenue'].mean())
holiday_lift_pct = (holiday_mean / non_holiday_mean - 1) * 100
holiday_ascii = (
    sales['holiday_name']
    .fillna('')
    .astype(str)
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('ascii')
)
tet_mask = holiday_ascii.str.contains('Tet|Giao thua', case=False, regex=True)
tet_mean = float(sales.loc[tet_mask, 'Revenue'].mean()) if tet_mask.any() else np.nan
tet_lift_pct = (tet_mean / non_holiday_mean - 1) * 100 if tet_mask.any() else np.nan

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
month_avg.sort_index().plot(kind='bar', ax=axes[0], color='#17becf')
axes[0].set_title('Average Revenue by Month')
axes[0].set_xlabel('Month')
dow_avg.plot(kind='bar', ax=axes[1], color='#9467bd')
axes[1].set_title('Average Revenue by Weekday')
axes[1].set_xlabel('Weekday')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_story_02_seasonality.png', dpi=150)
plt.show()

print(f'Peak month avg revenue: month {int(month_avg.index[0])} -> {month_avg.iloc[0]:,.0f}')
print(f'Trough month avg revenue: month {int(month_avg.index[-1])} -> {month_avg.iloc[-1]:,.0f}')
print(f'Holiday lift vs non-holiday: {holiday_lift_pct:.2f}%')
print(f'Tet window lift vs non-holiday: {tet_lift_pct:.2f}%')

## 3) Demand Channels and Geography

In [ ]:
region_tbl = (
    line[['order_id', 'net']]
    .merge(orders[['order_id', 'zip']], on='order_id', how='left')
    .merge(geography[['zip', 'region']], on='zip', how='left')
    .groupby('region')['net']
    .sum()
    .sort_values(ascending=False)
)
region_share = region_tbl / region_tbl.sum() * 100

web_daily = web.groupby('date').agg(
    sessions=('sessions', 'sum'),
    visitors=('unique_visitors', 'sum'),
    page_views=('page_views', 'sum'),
    bounce=('bounce_rate', 'mean'),
)
joint = sales.rename(columns={'Date': 'date'})[['date', 'Revenue']].merge(web_daily, on='date', how='inner').dropna()
web_corr = joint[['Revenue', 'sessions', 'visitors', 'page_views', 'bounce']].corr()['Revenue'].drop('Revenue').sort_values(ascending=False)

source_tbl = (
    orders[['order_id', 'order_source']]
    .merge(line[['order_id', 'net']], on='order_id', how='left')
    .groupby('order_source')
    .agg(orders=('order_id', 'nunique'), net=('net', 'sum'))
)
source_tbl['aov'] = source_tbl['net'] / source_tbl['orders']
source_tbl['net_share'] = source_tbl['net'] / source_tbl['net'].sum() * 100
source_tbl = source_tbl.sort_values('net', ascending=False)

display(pd.DataFrame({'region_net': region_tbl, 'region_share_pct': region_share}).round(2))
display(source_tbl.round(2))
display(web_corr.round(4).to_frame('corr_with_revenue'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
region_share.sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='#2ca02c')
axes[0].set_title('Revenue Share by Region (%)')
axes[0].set_ylabel('Share %')
source_tbl['net_share'].plot(kind='bar', ax=axes[1], color='#8c564b')
axes[1].set_title('Revenue Share by Order Source (%)')
axes[1].set_ylabel('Share %')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_story_03_geo_channel.png', dpi=150)
plt.show()

## 4) Promotion Economics: Volume vs Value vs Margin

In [ ]:
promo_stats = line.groupby('has_promo').agg(
    lines=('order_id', 'size'),
    qty=('quantity', 'sum'),
    net=('net', 'sum'),
    gross=('gross', 'sum'),
    discount=('discount_amount', 'sum'),
    margin_est=('margin_est', 'sum'),
)
promo_stats['discount_rate'] = promo_stats['discount'] / promo_stats['gross'] * 100
promo_stats['aov'] = promo_stats['net'] / promo_stats['lines']
promo_stats['margin_est_pct'] = promo_stats['margin_est'] / promo_stats['net'] * 100

line['discount_pct'] = np.where(line['gross'] > 0, line['discount_amount'] / line['gross'] * 100, 0)
line['discount_bucket'] = pd.cut(
    line['discount_pct'],
    bins=[-0.01, 0.01, 5, 10, 20, 100],
    labels=['0%', '0-5%', '5-10%', '10-20%', '20%+'],
)
bucket_tbl = line.groupby('discount_bucket', observed=False).agg(
    lines=('order_id', 'size'),
    net=('net', 'sum'),
    gross=('gross', 'sum'),
    discount=('discount_amount', 'sum'),
    margin_est=('margin_est', 'sum'),
)
bucket_tbl['aov'] = bucket_tbl['net'] / bucket_tbl['lines']
bucket_tbl['discount_rate'] = bucket_tbl['discount'] / bucket_tbl['gross'] * 100
bucket_tbl['margin_est_pct'] = bucket_tbl['margin_est'] / bucket_tbl['net'] * 100

promo_channel = (
    line[line['has_promo']]
    .merge(promotions[['promo_id', 'promo_channel']], on='promo_id', how='left')
    .groupby('promo_channel')
    .agg(lines=('order_id', 'size'), net=('net', 'sum'), gross=('gross', 'sum'), discount=('discount_amount', 'sum'))
)
promo_channel['discount_rate'] = promo_channel['discount'] / promo_channel['gross'] * 100
promo_channel['aov'] = promo_channel['net'] / promo_channel['lines']
promo_channel = promo_channel.sort_values('net', ascending=False)

display(promo_stats.round(2))
display(bucket_tbl.round(2))
display(promo_channel.round(2))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
promo_stats['aov'].rename({False: 'No promo', True: 'Promo'}).plot(kind='bar', ax=axes[0], color=['#1f77b4', '#ff7f0e'])
axes[0].set_title('AOV: Promo vs Non-promo')
axes[0].set_ylabel('AOV')
bucket_tbl['margin_est_pct'].plot(kind='bar', ax=axes[1], color='#d62728')
axes[1].set_title('Estimated Margin % by Discount Bucket')
axes[1].set_ylabel('Margin %')
promo_channel['discount_rate'].plot(kind='bar', ax=axes[2], color='#9467bd')
axes[2].set_title('Discount Rate by Promo Channel')
axes[2].set_ylabel('Discount %')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_story_04_promo_economics.png', dpi=150)
plt.show()

## 5) Customer Behavior and Lifecycle

In [ ]:
order_net = line.groupby('order_id')['net'].sum().reset_index(name='order_net')
cust_ord = (
    orders[['order_id', 'customer_id']]
    .merge(order_net, on='order_id', how='left')
    .merge(customers[['customer_id', 'age_group', 'acquisition_channel']], on='customer_id', how='left')
)

age_tbl = cust_ord.groupby('age_group').agg(
    orders=('order_id', 'nunique'),
    customers=('customer_id', 'nunique'),
    net=('order_net', 'sum'),
)
age_tbl['orders_per_customer'] = age_tbl['orders'] / age_tbl['customers']
age_tbl['aov'] = age_tbl['net'] / age_tbl['orders']
age_tbl = age_tbl.sort_values('orders_per_customer', ascending=False)

acq_tbl = cust_ord.groupby('acquisition_channel').agg(
    orders=('order_id', 'nunique'),
    customers=('customer_id', 'nunique'),
    net=('order_net', 'sum'),
)
acq_tbl['orders_per_customer'] = acq_tbl['orders'] / acq_tbl['customers']
acq_tbl['aov'] = acq_tbl['net'] / acq_tbl['orders']
acq_tbl = acq_tbl.sort_values('aov', ascending=False)

od = orders[['customer_id', 'order_date']].sort_values(['customer_id', 'order_date'])
first = od.groupby('customer_id').nth(0).rename(columns={'order_date': 'first_order'}).reset_index()
second = od.groupby('customer_id').nth(1).rename(columns={'order_date': 'second_order'}).reset_index()
cohort = first.merge(second, on='customer_id', how='left').merge(customers[['customer_id', 'age_group']], on='customer_id', how='left')
cohort['gap_days'] = (cohort['second_order'] - cohort['first_order']).dt.days
cohort['repeat_180'] = cohort['gap_days'].le(180)
cohort['repeat_365'] = cohort['gap_days'].le(365)
overall_repeat_180 = float(cohort['repeat_180'].mean() * 100)
overall_repeat_365 = float(cohort['repeat_365'].mean() * 100)
age_repeat = (cohort.groupby('age_group')['repeat_180'].mean() * 100).sort_values(ascending=False)

display(age_tbl.round(2))
display(acq_tbl.round(2))
display(age_repeat.round(2).to_frame('repeat_180_pct'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
age_tbl['orders_per_customer'].plot(kind='bar', ax=axes[0], color='#2ca02c')
axes[0].set_title('Orders per Customer by Age Group')
axes[0].set_ylabel('Orders / Customer')
age_repeat.plot(kind='bar', ax=axes[1], color='#17becf')
axes[1].set_title('Repeat <=180 days by Age Group (%)')
axes[1].set_ylabel('Percent')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_story_05_customer_lifecycle.png', dpi=150)
plt.show()

print(f'Overall repeat <=180 days: {overall_repeat_180:.2f}%')
print(f'Overall repeat <=365 days: {overall_repeat_365:.2f}%')

## 6) Operational Leakage: Returns, Cancellations, Delivery

In [ ]:
cancel_tbl = orders.assign(cancelled=orders['order_status'].eq('cancelled')).groupby('payment_method').agg(
    orders=('order_id', 'size'),
    cancelled=('cancelled', 'sum'),
)
cancel_tbl['cancel_rate_pct'] = cancel_tbl['cancelled'] / cancel_tbl['orders'] * 100
cancel_tbl['cancel_share_pct'] = cancel_tbl['cancelled'] / cancel_tbl['cancelled'].sum() * 100
cancel_tbl = cancel_tbl.sort_values('cancelled', ascending=False)

return_reason_tbl = (returns['return_reason'].value_counts(normalize=True) * 100).to_frame('pct').round(2)
ret_cat_qty = returns.merge(products[['product_id', 'category']], on='product_id', how='left').groupby('category')['return_quantity'].sum()
sold_cat_qty = line.groupby('category')['quantity'].sum()
ret_cat_rate = (ret_cat_qty / sold_cat_qty * 100).sort_values(ascending=False)

ret_size_qty = returns.merge(products[['product_id', 'size']], on='product_id', how='left').groupby('size')['return_quantity'].sum()
sold_size_qty = line.groupby('size')['quantity'].sum()
ret_size_rate = (ret_size_qty / sold_size_qty * 100).sort_values(ascending=False)

ship = shipments.copy()
ship['lead_days'] = (ship['delivery_date'] - ship['ship_date']).dt.days
ship_stats = ship['lead_days'].describe(percentiles=[0.5, 0.9, 0.95])
ship_rating = (
    ship.groupby('order_id')['lead_days'].mean().reset_index()
    .merge(reviews.groupby('order_id')['rating'].mean().reset_index(), on='order_id', how='inner')
)
ship_rating_corr = float(ship_rating['lead_days'].corr(ship_rating['rating']))

display(cancel_tbl.round(2))
display(return_reason_tbl)
display(ret_cat_rate.round(2).to_frame('return_rate_pct'))
display(ret_size_rate.round(2).to_frame('return_rate_pct'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cancel_tbl['cancel_rate_pct'].plot(kind='bar', ax=axes[0], color='#ff7f0e')
axes[0].set_title('Cancellation Rate by Payment Method (%)')
axes[0].set_ylabel('Cancel %')
ret_cat_rate.plot(kind='bar', ax=axes[1], color='#d62728')
axes[1].set_title('Return Rate by Category (%)')
axes[1].set_ylabel('Return %')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_story_06_operational_leakage.png', dpi=150)
plt.show()

print(f"Delivery lead-time median: {ship_stats['50%']:.2f} days")
print(f"Delivery lead-time p90: {ship_stats['90%']:.2f} days")
print(f"Delivery lead-time p95: {ship_stats['95%']:.2f} days")
print(f'Lead-time vs rating correlation: {ship_rating_corr:.4f}')

## 7) Inventory Pressure vs Revenue

In [ ]:
inv_month = inventory.groupby(inventory['snapshot_date'].dt.to_period('M')).agg(
    stockout_rate=('stockout_flag', 'mean'),
    overstock_rate=('overstock_flag', 'mean'),
    sell_through=('sell_through_rate', 'mean'),
    days_supply=('days_of_supply', 'mean'),
).reset_index()
inv_month['month'] = inv_month['snapshot_date'].dt.to_timestamp()
sales_month = sales.assign(month=sales['Date'].dt.to_period('M')).groupby('month')['Revenue'].sum().reset_index()
sales_month['month'] = sales_month['month'].dt.to_timestamp()
inv_joint = sales_month.merge(inv_month[['month', 'stockout_rate', 'overstock_rate', 'sell_through', 'days_supply']], on='month', how='inner')
inv_corr = inv_joint[['Revenue', 'stockout_rate', 'overstock_rate', 'sell_through', 'days_supply']].corr()['Revenue'].drop('Revenue').sort_values(ascending=False)

display(inv_corr.round(4).to_frame('corr_with_monthly_revenue'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.scatterplot(data=inv_joint, x='sell_through', y='Revenue', ax=axes[0], alpha=0.7)
axes[0].set_title('Revenue vs Sell-through')
sns.scatterplot(data=inv_joint, x='days_supply', y='Revenue', ax=axes[1], alpha=0.7, color='#8c564b')
axes[1].set_title('Revenue vs Days of Supply')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_story_07_inventory_pressure.png', dpi=150)
plt.show()

## 8) Executive Interpretation and Action Plan

In [ ]:
promo_line_share = float(line['has_promo'].mean() * 100)
promo_net_share = float(line.loc[line['has_promo'], 'net'].sum() / line['net'].sum() * 100)
promo_aov = float(line.loc[line['has_promo'], 'net'].mean())
nonpromo_aov = float(line.loc[~line['has_promo'], 'net'].mean())
promo_margin_pct = float((line.loc[line['has_promo'], 'margin_est'].sum() / line.loc[line['has_promo'], 'net'].sum()) * 100)
nonpromo_margin_pct = float((line.loc[~line['has_promo'], 'margin_est'].sum() / line.loc[~line['has_promo'], 'net'].sum()) * 100)

east_share = float((
    line[['order_id', 'net']]
    .merge(orders[['order_id', 'zip']], on='order_id', how='left')
    .merge(geography[['zip', 'region']], on='zip', how='left')
    .groupby('region')['net']
    .sum()
    .pipe(lambda s: s / s.sum() * 100)
)['East'])

summary_md = f'''# EDA Business Story Summary

## Financial Core
- Total revenue: **{total_revenue:,.0f}**
- Total COGS: **{total_cogs:,.0f}**
- Gross margin: **{gross_margin_pct:.2f}%**
- End-to-end revenue CAGR: **{cagr * 100:.2f}%**

## Demand Shape
- Peak month average revenue: **{month_avg.iloc[0]:,.0f}** (month {int(month_avg.index[0])})
- Lowest month average revenue: **{month_avg.iloc[-1]:,.0f}** (month {int(month_avg.index[-1])})
- Holiday lift vs non-holiday days: **{holiday_lift_pct:.2f}%**
- Tet-window lift vs non-holiday days: **{tet_lift_pct:.2f}%**

## Channel and Geo
- East contributes **{east_share:.2f}%** of net revenue.
- Web sessions correlation with revenue: **{web_corr['sessions']:.4f}** (moderate positive signal).

## Promotion Economics
- Promo line share: **{promo_line_share:.2f}%**, but promo net-revenue share is only **{promo_net_share:.2f}%**.
- Promo AOV: **{promo_aov:,.0f}** vs non-promo AOV **{nonpromo_aov:,.0f}**.
- Estimated margin with promo: **{promo_margin_pct:.2f}%** vs non-promo **{nonpromo_margin_pct:.2f}%**.

## Operations
- Highest cancellation volume payment method: **{cancel_tbl.index[0]}** ({int(cancel_tbl.iloc[0]['cancelled']):,} cancelled orders).
- Highest cancellation rate payment method: **{cancel_tbl['cancel_rate_pct'].idxmax()}** ({cancel_tbl['cancel_rate_pct'].max():.2f}%).
- Top return reason: **{return_reason_tbl.index[0]}** ({return_reason_tbl.iloc[0, 0]:.2f}% of return records).
- Delivery lead-time median/p90/p95: **{ship_stats['50%']:.1f}/{ship_stats['90%']:.1f}/{ship_stats['95%']:.1f} days**.

## Customer Lifecycle
- Repeat <= 180 days: **{overall_repeat_180:.2f}%**.
- Repeat <= 365 days: **{overall_repeat_365:.2f}%**.
- Highest orders-per-customer age group: **{age_tbl.index[0]}** ({age_tbl.iloc[0]['orders_per_customer']:.2f}).

## Priority Business Actions
1. **Tighten discount governance**: cap high-discount campaigns and prioritize channels with better value retention.
2. **Geo-focused inventory allocation**: protect East availability while reducing overstock in lower-yield periods.
3. **Payment and checkout optimization**: reduce cancellation leakage for high-risk payment paths.
4. **Returns prevention**: focus on top return reasons and high-return categories/sizes with fit and quality interventions.
5. **Lifecycle orchestration**: trigger re-engagement before 180-day churn windows for high-frequency cohorts.
'''

display(Markdown(summary_md))

(OUT_DIR / 'eda_business_story.md').write_text(summary_md, encoding='utf-8')
print('Saved:', OUT_DIR / 'eda_business_story.md')

kpi_payload = {
    'total_revenue': total_revenue,
    'total_cogs': total_cogs,
    'gross_margin_pct': gross_margin_pct,
    'cagr_pct': cagr * 100,
    'promo_line_share_pct': promo_line_share,
    'promo_net_share_pct': promo_net_share,
    'promo_aov': promo_aov,
    'nonpromo_aov': nonpromo_aov,
    'promo_margin_est_pct': promo_margin_pct,
    'nonpromo_margin_est_pct': nonpromo_margin_pct,
    'holiday_lift_pct': holiday_lift_pct,
    'tet_lift_pct': tet_lift_pct,
    'repeat_180_pct': overall_repeat_180,
    'repeat_365_pct': overall_repeat_365,
    'east_revenue_share_pct': east_share,
}

with open(OUT_DIR / 'eda_business_kpis.json', 'w', encoding='utf-8') as f:
    json.dump(kpi_payload, f, indent=2)
print('Saved:', OUT_DIR / 'eda_business_kpis.json')

yearly.round(4).to_csv(OUT_DIR / 'kpi_yearly_financials.csv')
source_tbl.round(4).to_csv(OUT_DIR / 'kpi_source_table.csv')
cancel_tbl.round(4).to_csv(OUT_DIR / 'kpi_cancellation_table.csv')
age_tbl.round(4).to_csv(OUT_DIR / 'kpi_age_table.csv')
bucket_tbl.round(4).to_csv(OUT_DIR / 'kpi_discount_bucket_table.csv')
print('Saved KPI tables to', OUT_DIR)